# GuardPlant — 70-Species Offline Classification Benchmark

This Google Colab notebook trains and compares exactly two image classifiers for the GuardPlant Android application:

1. **Ultralytics YOLO26 Nano Classifier** (`yolo26n-cls.pt`)
2. **Torchvision EfficientNet-B0** with ImageNet pretrained weights

The dataset contains approximately 28,000 images across 70 plant species and is already split into `train`, `val`, and `test` directories. Validation data is used for early stopping and model selection; the held-out test split is used only for the final benchmark. A canonical alphabetical class mapping is saved to `class_names.json` and reused by training, evaluation, export, and Android integration.

The notebook supports two workflows after setup: train fresh models, or disable training in the configuration cell and load previously trained checkpoints from Google Drive. Both paths populate the same model and history variables used by evaluation and export.


## 1. Environment and global configuration


In [ ]:
%pip install -q "ultralytics>=8.4.83,<9" "onnx>=1.16,<2" "tabulate>=0.9,<1"


In [ ]:
import json
import os
import random
import shutil
import time
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnx
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Markdown, display
from PIL import Image
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from torchvision.transforms import InterpolationMode
from tqdm.auto import tqdm
from ultralytics import YOLO


In [ ]:
# Core experiment configuration
SEED = 42
IMAGE_SIZE = 224
MAX_EPOCHS = 100
PATIENCE = 15
EXPECTED_NUM_CLASSES = 70

YOLO_BATCH = -1
EFFICIENTNET_BATCH_SIZE = 64

# Workflow switches: set either training flag to False to use its Drive-loading cell instead.
TRAIN_YOLO26 = True
TRAIN_EFFICIENTNET = True
EXPORT_LITERT = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Google Drive, extraction, and dataset validation

The expected archive is `/content/drive/MyDrive/dataset.zip`. Extraction is skipped when a valid split layout already exists.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

ZIP_PATH = Path("/content/drive/MyDrive/dataset.zip")
EXTRACT_DIR = Path("/content/dataset")
REQUIRED_SPLITS = ("train", "val", "test")
FORCE_REEXTRACT = False

if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {ZIP_PATH}")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
possible_roots = (EXTRACT_DIR, EXTRACT_DIR / "dataset")
splits_exist = any(all((root / split).is_dir() for split in REQUIRED_SPLITS) for root in possible_roots)

if FORCE_REEXTRACT or not splits_exist:
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise zipfile.BadZipFile(f"CRC failure in archive member: {bad_member}")
        archive.extractall(EXTRACT_DIR)
    print(f"Extracted {ZIP_PATH} into {EXTRACT_DIR}")
else:
    print("Existing dataset splits found; extraction skipped.")

DATASET_ROOT = next(
    (root for root in possible_roots if all((root / split).is_dir() for split in REQUIRED_SPLITS)),
    None,
)
if DATASET_ROOT is None:
    raise FileNotFoundError(f"Could not find train/val/test below {EXTRACT_DIR}")
print(f"Dataset root: {DATASET_ROOT}")


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp"}
CLASS_NAMES_PATH = Path.cwd() / "class_names.json"
DATASET_STATS_PATH = Path.cwd() / "dataset_stats.csv"


def class_directories(split_dir):
    return sorted((path for path in split_dir.iterdir() if path.is_dir()), key=lambda path: path.name)


def count_images(directory):
    return sum(
        path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        for path in directory.rglob("*")
    )


split_classes = {}
dataset_stat_rows = []
for split in REQUIRED_SPLITS:
    directories = class_directories(DATASET_ROOT / split)
    split_classes[split] = [path.name for path in directories]
    for class_index, directory in enumerate(directories):
        image_count = count_images(directory)
        if image_count == 0:
            raise ValueError(f"Empty class folder: {directory}")
        dataset_stat_rows.append(
            {"split": split, "class_index": class_index, "class_id": directory.name, "image_count": image_count}
        )

class_names = split_classes["train"]
if len(class_names) != EXPECTED_NUM_CLASSES:
    raise ValueError(f"Expected {EXPECTED_NUM_CLASSES} classes, found {len(class_names)} in train.")
for split in REQUIRED_SPLITS:
    if split_classes[split] != class_names:
        missing = sorted(set(class_names) - set(split_classes[split]))
        extra = sorted(set(split_classes[split]) - set(class_names))
        raise ValueError(f"Class mismatch in {split}: missing={missing}, extra={extra}")

with CLASS_NAMES_PATH.open("w", encoding="utf-8") as file:
    json.dump(class_names, file, ensure_ascii=False, indent=2)
dataset_stats_df = pd.DataFrame(dataset_stat_rows)
dataset_stats_df.to_csv(DATASET_STATS_PATH, index=False)

print(f"Validated identical alphabetical indexing for {len(class_names)} classes.")
for split in REQUIRED_SPLITS:
    total = dataset_stats_df.loc[dataset_stats_df["split"] == split, "image_count"].sum()
    print(f"{split:>5}: {total:,} images")
print(f"Saved: {CLASS_NAMES_PATH}")
print(f"Saved: {DATASET_STATS_PATH}")


## 3. EfficientNet datasets and canonical test manifest

EfficientNet validation/test preprocessing comes directly from `EfficientNet_B0_Weights.DEFAULT.transforms()`. The canonical test manifest stores the exact image paths and labels later passed to both native pipelines.


In [ ]:
EFFICIENTNET_WEIGHTS = EfficientNet_B0_Weights.DEFAULT
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

efficientnet_train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(IMAGE_SIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)
efficientnet_eval_transform = EFFICIENTNET_WEIGHTS.transforms()

train_dataset = datasets.ImageFolder(DATASET_ROOT / "train", transform=efficientnet_train_transform)
val_dataset = datasets.ImageFolder(DATASET_ROOT / "val", transform=efficientnet_eval_transform)
test_dataset = datasets.ImageFolder(DATASET_ROOT / "test", transform=efficientnet_eval_transform)
expected_mapping = {name: index for index, name in enumerate(class_names)}
for split, dataset in (("train", train_dataset), ("val", val_dataset), ("test", test_dataset)):
    if dataset.class_to_idx != expected_mapping:
        raise RuntimeError(f"ImageFolder mapping mismatch in {split}.")

NUM_WORKERS = min(4, os.cpu_count() or 1)
loader_kwargs = {
    "batch_size": EFFICIENTNET_BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",
    "persistent_workers": NUM_WORKERS > 0,
}
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, shuffle=True, generator=loader_generator, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

test_manifest = [(Path(path), int(label)) for path, label in test_dataset.samples]
if len(test_manifest) != len(test_dataset) or not all(path.is_file() for path, _ in test_manifest):
    raise RuntimeError("Canonical test manifest is incomplete.")
YOLO_DATASET_ROOT = str(DATASET_ROOT)

sample_images, sample_labels = next(iter(train_loader))
assert sample_images.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)
print(f"Batches — train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")
print(f"Canonical held-out test files: {len(test_manifest):,}")


## 4. Shared artifact paths and workflow state


In [ ]:
ARTIFACT_ROOT = Path.cwd() / "guardplant_artifacts"
PLOTS_DIR = Path.cwd() / "plots"
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/GuardPlant_Artifacts")
for directory in (ARTIFACT_ROOT, PLOTS_DIR, DRIVE_ARTIFACT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

YOLO_BEST_PATH = Path.cwd() / "yolo26n_best.pt"
EFFICIENTNET_BEST_PATH = Path.cwd() / "efficientnet_b0_best.pth"
YOLO_HISTORY_PATH = ARTIFACT_ROOT / "yolo26n_history.csv"
YOLO_METADATA_PATH = ARTIFACT_ROOT / "yolo26n_metadata.json"
EFFICIENTNET_HISTORY_PATH = ARTIFACT_ROOT / "efficientnet_b0_history.json"

yolo_model = None
efficientnet_model = None
yolo_history = pd.DataFrame()
efficientnet_history = {}
yolo_training_summary = {}
efficientnet_training_summary = {}


def copy_to_drive(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    destination = DRIVE_ARTIFACT_DIR / path.name
    shutil.copy2(path, destination)
    return destination


def load_trusted_checkpoint(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


# Model A — YOLO26n-cls

Run the training cell when `TRAIN_YOLO26=True`. Otherwise skip training automatically and use the following Drive-loading cell.


In [ ]:
def summarize_yolo_history(history_frame):
    if history_frame.empty or "metrics/accuracy_top1" not in history_frame.columns:
        return {"best_epoch": np.nan, "best_val_top1": np.nan}
    values = pd.to_numeric(history_frame["metrics/accuracy_top1"], errors="coerce")
    best_row_index = values.idxmax()
    best_value = float(values.loc[best_row_index])
    if best_value <= 1.0001:
        best_value *= 100.0
    epoch_values = pd.to_numeric(history_frame["epoch"], errors="coerce")
    best_epoch = int(epoch_values.loc[best_row_index])
    if epoch_values.min() == 0:
        best_epoch += 1
    return {"best_epoch": best_epoch, "best_val_top1": best_value}


def history_from_yolo_checkpoint(model):
    checkpoint = getattr(model, "ckpt", None)
    if not isinstance(checkpoint, dict) or not isinstance(checkpoint.get("train_results"), dict):
        return pd.DataFrame()
    history_frame = pd.DataFrame(checkpoint["train_results"])
    history_frame.columns = history_frame.columns.str.strip()
    return history_frame


if TRAIN_YOLO26:
    yolo_model = YOLO("yolo26n-cls.pt")
    yolo_model.train(
        data=YOLO_DATASET_ROOT,
        epochs=MAX_EPOCHS,
        patience=PATIENCE,
        imgsz=IMAGE_SIZE,
        batch=YOLO_BATCH,
        optimizer="auto",
        seed=SEED,
        deterministic=True,
        workers=NUM_WORKERS,
        device=0 if DEVICE.type == "cuda" else "cpu",
        project=str(ARTIFACT_ROOT / "yolo"),
        name="yolo26n_cls",
        exist_ok=True,
        save=True,
        plots=True,
    )
    yolo_run_dir = Path(yolo_model.trainer.save_dir)
    native_best_path = yolo_run_dir / "weights" / "best.pt"
    native_history_path = yolo_run_dir / "results.csv"
    if not native_best_path.is_file() or not native_history_path.is_file():
        raise FileNotFoundError("YOLO training did not produce the expected best checkpoint and history.")
    shutil.copy2(native_best_path, YOLO_BEST_PATH)
    shutil.copy2(native_history_path, YOLO_HISTORY_PATH)
    yolo_model = YOLO(str(YOLO_BEST_PATH))
    yolo_history = pd.read_csv(YOLO_HISTORY_PATH)
    yolo_history.columns = yolo_history.columns.str.strip()
    yolo_training_summary = summarize_yolo_history(yolo_history)
    with YOLO_METADATA_PATH.open("w", encoding="utf-8") as file:
        json.dump(yolo_training_summary, file, indent=2)
    for path in (YOLO_BEST_PATH, YOLO_HISTORY_PATH, YOLO_METADATA_PATH):
        print(f"Backed up: {copy_to_drive(path)}")
    print(f"YOLO best checkpoint: {YOLO_BEST_PATH}")
else:
    print("YOLO training disabled; run the next cell to load the saved checkpoint.")


### Optional alternative: load YOLO26 from Google Drive

Set `TRAIN_YOLO26=False` in the configuration cell. The history and metadata sidecars are loaded when available so plots and benchmark metadata remain available without retraining.


In [ ]:
YOLO_SAVED_MODEL_PATH = "/content/drive/MyDrive/GuardPlant_Artifacts/yolo26n_best.pt"
YOLO_SAVED_HISTORY_PATH = "/content/drive/MyDrive/GuardPlant_Artifacts/yolo26n_history.csv"
YOLO_SAVED_METADATA_PATH = "/content/drive/MyDrive/GuardPlant_Artifacts/yolo26n_metadata.json"

if not TRAIN_YOLO26:
    saved_model_path = Path(YOLO_SAVED_MODEL_PATH)
    if not saved_model_path.is_file():
        raise FileNotFoundError(saved_model_path)
    shutil.copy2(saved_model_path, YOLO_BEST_PATH)
    yolo_model = YOLO(str(YOLO_BEST_PATH))
    if Path(YOLO_SAVED_HISTORY_PATH).is_file():
        shutil.copy2(YOLO_SAVED_HISTORY_PATH, YOLO_HISTORY_PATH)
        yolo_history = pd.read_csv(YOLO_HISTORY_PATH)
        yolo_history.columns = yolo_history.columns.str.strip()
        yolo_training_summary = summarize_yolo_history(yolo_history)
    else:
        yolo_history = history_from_yolo_checkpoint(yolo_model)
        if not yolo_history.empty:
            yolo_history.to_csv(YOLO_HISTORY_PATH, index=False)
            yolo_training_summary = summarize_yolo_history(yolo_history)
    if Path(YOLO_SAVED_METADATA_PATH).is_file():
        shutil.copy2(YOLO_SAVED_METADATA_PATH, YOLO_METADATA_PATH)
        with Path(YOLO_SAVED_METADATA_PATH).open("r", encoding="utf-8") as file:
            yolo_training_summary.update(json.load(file))
    if not yolo_training_summary:
        warnings.warn("YOLO history metadata is unavailable; best epoch/validation metrics will be unknown.")
        yolo_training_summary = {"best_epoch": np.nan, "best_val_top1": np.nan}
    print(f"Loaded YOLO checkpoint: {YOLO_BEST_PATH}")


# Model B — EfficientNet-B0


In [ ]:
def build_efficientnet(pretrained):
    model = efficientnet_b0(weights=EFFICIENTNET_WEIGHTS if pretrained else None)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, EXPECTED_NUM_CLASSES)
    return model


def make_grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def run_efficientnet_epoch(model, data_loader, loss_function, device, optimizer=None, scaler=None, phase="train"):
    training = optimizer is not None
    model.train(training)
    loss_sum = 0.0
    top1_correct = 0
    top5_correct = 0
    sample_count = 0
    amp_enabled = device.type == "cuda"

    for images, targets in tqdm(data_loader, desc=phase, leave=False):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                logits = model(images)
                loss = loss_function(logits, targets)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite {phase} loss: {loss.item()}")
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        top5 = logits.topk(5, dim=1).indices
        batch_size = targets.size(0)
        sample_count += batch_size
        loss_sum += loss.item() * batch_size
        top1_correct += top5[:, 0].eq(targets).sum().item()
        top5_correct += top5.eq(targets.unsqueeze(1)).any(1).sum().item()
    return {
        "loss": loss_sum / sample_count,
        "top1": 100.0 * top1_correct / sample_count,
        "top5": 100.0 * top5_correct / sample_count,
    }


In [ ]:
if TRAIN_EFFICIENTNET:
    efficientnet_model = build_efficientnet(pretrained=True).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(efficientnet_model.parameters(), lr=1e-3)
    scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    scaler = make_grad_scaler(DEVICE.type == "cuda")
    efficientnet_history = {
        "train_loss": [], "val_loss": [], "train_top1": [], "val_top1": [],
        "train_top5": [], "val_top5": [], "learning_rate": []
    }
    best_val_top1 = float("-inf")
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"\nEpoch {epoch:03d}/{MAX_EPOCHS} | lr={current_lr:.6f}")
        train_metrics = run_efficientnet_epoch(
            efficientnet_model, train_loader, criterion, DEVICE, optimizer, scaler, "train"
        )
        val_metrics = run_efficientnet_epoch(
            efficientnet_model, val_loader, criterion, DEVICE, phase="val"
        )
        for metric in ("loss", "top1", "top5"):
            efficientnet_history[f"train_{metric}"].append(train_metrics[metric])
            efficientnet_history[f"val_{metric}"].append(val_metrics[metric])
        efficientnet_history["learning_rate"].append(current_lr)
        scheduler.step()

        improved = val_metrics["top1"] > best_val_top1
        if improved:
            best_val_top1 = val_metrics["top1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "epoch": epoch,
                    "best_val_top1": best_val_top1,
                    "model_state_dict": efficientnet_model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "class_names": class_names,
                    "history": efficientnet_history,
                    "image_size": IMAGE_SIZE,
                },
                EFFICIENTNET_BEST_PATH,
            )
        else:
            epochs_without_improvement += 1

        with EFFICIENTNET_HISTORY_PATH.open("w", encoding="utf-8") as file:
            json.dump(efficientnet_history, file, indent=2)
        print(
            f"train loss={train_metrics['loss']:.4f}, top1={train_metrics['top1']:.2f}%, top5={train_metrics['top5']:.2f}% | "
            f"val loss={val_metrics['loss']:.4f}, top1={val_metrics['top1']:.2f}%, top5={val_metrics['top5']:.2f}%"
        )
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping at epoch {epoch}; no validation Top-1 improvement for {PATIENCE} epochs.")
            break

    best_checkpoint = load_trusted_checkpoint(EFFICIENTNET_BEST_PATH, DEVICE)
    best_checkpoint["history"] = efficientnet_history
    torch.save(best_checkpoint, EFFICIENTNET_BEST_PATH)
    efficientnet_model.load_state_dict(best_checkpoint["model_state_dict"])
    efficientnet_model.eval()
    efficientnet_training_summary = {"best_epoch": best_epoch, "best_val_top1": best_val_top1}
    for path in (EFFICIENTNET_BEST_PATH, EFFICIENTNET_HISTORY_PATH):
        print(f"Backed up: {copy_to_drive(path)}")
    print(f"EfficientNet best checkpoint: {EFFICIENTNET_BEST_PATH}")
else:
    print("EfficientNet training disabled; run the next cell to load the saved checkpoint.")


### Optional alternative: load EfficientNet-B0 from Google Drive

Set `TRAIN_EFFICIENTNET=False` in the configuration cell. The architecture is recreated with the same 70-class head before the saved state dictionary is loaded.


In [ ]:
EFFICIENTNET_SAVED_MODEL_PATH = "/content/drive/MyDrive/GuardPlant_Artifacts/efficientnet_b0_best.pth"
EFFICIENTNET_SAVED_HISTORY_PATH = "/content/drive/MyDrive/GuardPlant_Artifacts/efficientnet_b0_history.json"

if not TRAIN_EFFICIENTNET:
    saved_model_path = Path(EFFICIENTNET_SAVED_MODEL_PATH)
    if not saved_model_path.is_file():
        raise FileNotFoundError(saved_model_path)
    shutil.copy2(saved_model_path, EFFICIENTNET_BEST_PATH)
    checkpoint = load_trusted_checkpoint(EFFICIENTNET_BEST_PATH, DEVICE)
    if checkpoint.get("class_names") != class_names:
        raise RuntimeError("Saved EfficientNet class mapping does not match class_names.json.")
    efficientnet_model = build_efficientnet(pretrained=False).to(DEVICE)
    efficientnet_model.load_state_dict(checkpoint["model_state_dict"])
    efficientnet_model.eval()
    efficientnet_history = checkpoint.get("history", {})
    if Path(EFFICIENTNET_SAVED_HISTORY_PATH).is_file():
        shutil.copy2(EFFICIENTNET_SAVED_HISTORY_PATH, EFFICIENTNET_HISTORY_PATH)
        with EFFICIENTNET_HISTORY_PATH.open("r", encoding="utf-8") as file:
            efficientnet_history = json.load(file)
    elif efficientnet_history:
        with EFFICIENTNET_HISTORY_PATH.open("w", encoding="utf-8") as file:
            json.dump(efficientnet_history, file, indent=2)
    efficientnet_training_summary = {
        "best_epoch": int(checkpoint["epoch"]),
        "best_val_top1": float(checkpoint["best_val_top1"]),
    }
    print(f"Loaded EfficientNet checkpoint: {EFFICIENTNET_BEST_PATH}")


# Evaluation and comparative benchmark

Both models are evaluated only after their best checkpoints are active. YOLO receives the canonical test file paths through Ultralytics prediction, including its native classification transforms. EfficientNet reads the same ordered paths through the weights-native Torchvision transform in `test_loader`.


In [ ]:
if yolo_model is None or efficientnet_model is None:
    raise RuntimeError("Train or load both models before evaluation.")
yolo_names = [yolo_model.names[index] for index in range(len(yolo_model.names))]
if yolo_names != class_names:
    raise RuntimeError("YOLO class order does not match class_names.json.")


def metrics_from_scores(scores, targets):
    top5 = scores.topk(5, dim=1).indices
    return (
        100.0 * top5[:, 0].eq(targets).float().mean().item(),
        100.0 * top5.eq(targets.unsqueeze(1)).any(1).float().mean().item(),
        top5[:, 0],
    )


@torch.inference_mode()
def evaluate_efficientnet(model, data_loader, device):
    score_batches, target_batches = [], []
    model.eval()
    for images, targets in tqdm(data_loader, desc="EfficientNet test"):
        logits = model(images.to(device, non_blocking=True))
        score_batches.append(logits.cpu())
        target_batches.append(targets.cpu())
    scores = torch.cat(score_batches)
    targets = torch.cat(target_batches)
    top1, top5, predictions = metrics_from_scores(scores, targets)
    return {"top1": top1, "top5": top5}, targets.numpy(), predictions.numpy()


def evaluate_yolo_native(model, manifest, batch_size):
    all_scores, all_targets = [], []
    for start in tqdm(range(0, len(manifest), batch_size), desc="YOLO native test"):
        chunk = manifest[start : start + batch_size]
        paths = [str(path) for path, _ in chunk]
        results = model.predict(
            source=paths,
            imgsz=IMAGE_SIZE,
            batch=batch_size,
            device=0 if DEVICE.type == "cuda" else "cpu",
            verbose=False,
        )
        if len(results) != len(chunk):
            raise RuntimeError("YOLO returned a different number of predictions than input paths.")
        all_scores.append(torch.stack([result.probs.data.cpu() for result in results]))
        all_targets.extend(label for _, label in chunk)
    scores = torch.cat(all_scores)
    targets = torch.tensor(all_targets, dtype=torch.long)
    top1, top5, predictions = metrics_from_scores(scores, targets)
    return {"top1": top1, "top5": top5}, targets.numpy(), predictions.numpy()


efficientnet_test_metrics, efficientnet_targets, efficientnet_predictions = evaluate_efficientnet(
    efficientnet_model, test_loader, DEVICE
)
yolo_test_metrics, yolo_targets, yolo_predictions = evaluate_yolo_native(
    yolo_model, test_manifest, EFFICIENTNET_BATCH_SIZE
)
manifest_targets = np.asarray([label for _, label in test_manifest])
if not (np.array_equal(efficientnet_targets, manifest_targets) and np.array_equal(yolo_targets, manifest_targets)):
    raise RuntimeError("Model evaluations did not use the same test files and label order.")
print("Fair held-out evaluation complete.")


## Development latency and benchmark table

Latency below is **Colab development pipeline latency**, not Android/mobile performance. It includes image decoding, each model's native preprocessing, and batch-size-one inference on the active runtime.


In [ ]:
def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


@torch.inference_mode()
def efficientnet_pipeline_call(path):
    with Image.open(path) as image:
        tensor = efficientnet_eval_transform(image.convert("RGB")).unsqueeze(0).to(DEVICE)
    return efficientnet_model(tensor)


def yolo_pipeline_call(path):
    return yolo_model.predict(
        source=str(path), imgsz=IMAGE_SIZE, batch=1,
        device=0 if DEVICE.type == "cuda" else "cpu", verbose=False
    )


def benchmark_pipeline(callable_fn, paths, warmup_count=3):
    for path in paths[:warmup_count]:
        callable_fn(path)
    timings = []
    for path in paths:
        synchronize_device()
        start = time.perf_counter()
        callable_fn(path)
        synchronize_device()
        timings.append((time.perf_counter() - start) * 1000.0)
    return float(np.mean(timings))


latency_count = min(30, len(test_manifest))
latency_indices = np.linspace(0, len(test_manifest) - 1, latency_count, dtype=int)
latency_paths = [test_manifest[index][0] for index in latency_indices]
yolo_latency_ms = benchmark_pipeline(yolo_pipeline_call, latency_paths)
efficientnet_latency_ms = benchmark_pipeline(efficientnet_pipeline_call, latency_paths)

def file_size_mb(path):
    return Path(path).stat().st_size / (1024 ** 2)


benchmark_df = pd.DataFrame(
    [
        {
            "Model": "YOLO26n-cls",
            "Test Top-1 (%)": yolo_test_metrics["top1"],
            "Test Top-5 (%)": yolo_test_metrics["top5"],
            "Best Validation Top-1 (%)": yolo_training_summary.get("best_val_top1", np.nan),
            "Best Epoch": yolo_training_summary.get("best_epoch", np.nan),
            "Parameters": sum(parameter.numel() for parameter in yolo_model.model.parameters()),
            "Checkpoint Size (MB)": file_size_mb(YOLO_BEST_PATH),
            "Colab Development Latency (ms/image)": yolo_latency_ms,
        },
        {
            "Model": "EfficientNet-B0",
            "Test Top-1 (%)": efficientnet_test_metrics["top1"],
            "Test Top-5 (%)": efficientnet_test_metrics["top5"],
            "Best Validation Top-1 (%)": efficientnet_training_summary["best_val_top1"],
            "Best Epoch": efficientnet_training_summary["best_epoch"],
            "Parameters": sum(parameter.numel() for parameter in efficientnet_model.parameters()),
            "Checkpoint Size (MB)": file_size_mb(EFFICIENTNET_BEST_PATH),
            "Colab Development Latency (ms/image)": efficientnet_latency_ms,
        },
    ]
).round(4)
BENCHMARK_RESULTS_PATH = Path.cwd() / "benchmark_results.csv"
benchmark_df.to_csv(BENCHMARK_RESULTS_PATH, index=False)
display(Markdown(benchmark_df.to_markdown(index=False)))
print(f"Saved: {BENCHMARK_RESULTS_PATH}")


# Per-class diagnostics and plots

The confusion matrices include all 70 classes; no class is selected or excluded by support. The worst-class diagnostic independently ranks each model's 15 lowest-recall classes.


In [ ]:
PER_CLASS_METRICS_PATH = Path.cwd() / "per_class_metrics.csv"
TOP_CONFUSIONS_PATH = Path.cwd() / "top_confusions.csv"
per_class_rows = []
top_confusion_rows = []
prediction_sets = (
    ("YOLO26n-cls", yolo_targets, yolo_predictions),
    ("EfficientNet-B0", efficientnet_targets, efficientnet_predictions),
)

for model_name, targets, predictions in prediction_sets:
    precision, recall, f1, support = precision_recall_fscore_support(
        targets, predictions, labels=np.arange(EXPECTED_NUM_CLASSES), zero_division=0
    )
    for class_index, class_id in enumerate(class_names):
        per_class_rows.append(
            {
                "Model": model_name, "Class Index": class_index, "Class ID": class_id,
                "Support": int(support[class_index]), "Precision (%)": 100.0 * precision[class_index],
                "Recall (%)": 100.0 * recall[class_index], "F1 (%)": 100.0 * f1[class_index],
            }
        )
    counts = confusion_matrix(targets, predictions, labels=np.arange(EXPECTED_NUM_CLASSES))
    for true_index, predicted_index in np.argwhere(counts > 0):
        if true_index == predicted_index:
            continue
        top_confusion_rows.append(
            {
                "Model": model_name, "True Class": class_names[true_index],
                "Predicted Class": class_names[predicted_index],
                "Count": int(counts[true_index, predicted_index]),
                "Fraction of True Class (%)": 100.0 * counts[true_index, predicted_index] / counts[true_index].sum(),
            }
        )

per_class_metrics_df = pd.DataFrame(per_class_rows).round(4)
top_confusion_columns = [
    "Model", "True Class", "Predicted Class", "Count", "Fraction of True Class (%)"
]
top_confusions_df = pd.DataFrame(top_confusion_rows, columns=top_confusion_columns)
if not top_confusions_df.empty:
    top_confusions_df = top_confusions_df.sort_values(
        ["Model", "Count", "Fraction of True Class (%)"], ascending=[True, False, False]
    )
top_confusions_df = top_confusions_df.round(4)
per_class_metrics_df.to_csv(PER_CLASS_METRICS_PATH, index=False)
top_confusions_df.to_csv(TOP_CONFUSIONS_PATH, index=False)
print(f"Saved: {PER_CLASS_METRICS_PATH}")
print(f"Saved: {TOP_CONFUSIONS_PATH}")


In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

if efficientnet_history:
    epochs = np.arange(1, len(efficientnet_history["train_loss"]) + 1)
    axes[0].plot(epochs, efficientnet_history["train_loss"], label="EfficientNet train")
    axes[0].plot(epochs, efficientnet_history["val_loss"], label="EfficientNet val")
    axes[1].plot(epochs, efficientnet_history["train_top1"], label="EfficientNet train")
    axes[1].plot(epochs, efficientnet_history["val_top1"], label="EfficientNet val")
if not yolo_history.empty:
    yolo_epochs = pd.to_numeric(yolo_history["epoch"]).to_numpy()
    if yolo_epochs.min() == 0:
        yolo_epochs += 1
    if "train/loss" in yolo_history:
        axes[0].plot(yolo_epochs, yolo_history["train/loss"], label="YOLO train")
    if "val/loss" in yolo_history:
        axes[0].plot(yolo_epochs, yolo_history["val/loss"], label="YOLO val")
    if "metrics/accuracy_top1" in yolo_history:
        yolo_accuracy = pd.to_numeric(yolo_history["metrics/accuracy_top1"]).to_numpy()
        if np.nanmax(yolo_accuracy) <= 1.0001:
            yolo_accuracy *= 100.0
        axes[1].plot(yolo_epochs, yolo_accuracy, label="YOLO val")
axes[0].set(title="Training and validation loss", xlabel="Epoch", ylabel="Loss")
axes[1].set(title="Top-1 accuracy", xlabel="Epoch", ylabel="Accuracy (%)", ylim=(0, 100))
for axis in axes:
    axis.legend()
fig.tight_layout()
TRAINING_CURVES_PATH = PLOTS_DIR / "training_curves.png"
fig.savefig(TRAINING_CURVES_PATH, dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
CONFUSION_PLOT_PATHS = {}
for model_name, targets, predictions in prediction_sets:
    normalized_matrix = confusion_matrix(
        targets, predictions, labels=np.arange(EXPECTED_NUM_CLASSES), normalize="true"
    )
    fig, axis = plt.subplots(figsize=(24, 20))
    sns.heatmap(
        normalized_matrix, ax=axis, cmap="Blues", vmin=0, vmax=1,
        xticklabels=class_names, yticklabels=class_names, cbar_kws={"label": "Recall fraction"}
    )
    axis.set(title=f"{model_name} — normalized 70x70 confusion matrix", xlabel="Predicted class", ylabel="True class")
    axis.tick_params(axis="x", labelrotation=90, labelsize=5)
    axis.tick_params(axis="y", labelrotation=0, labelsize=5)
    file_stem = "yolo26n" if model_name == "YOLO26n-cls" else "efficientnet_b0"
    output_path = PLOTS_DIR / f"{file_stem}_confusion_matrix.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()
    CONFUSION_PLOT_PATHS[model_name] = output_path


In [ ]:
worst_recall_df = (
    per_class_metrics_df.sort_values(["Model", "Recall (%)"], ascending=[True, True])
    .groupby("Model", group_keys=False)
    .head(15)
)
display(Markdown(worst_recall_df.to_markdown(index=False)))

fig, axes = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)
for axis, model_name in zip(axes, ("YOLO26n-cls", "EfficientNet-B0")):
    subset = worst_recall_df[worst_recall_df["Model"] == model_name].sort_values("Recall (%)")
    axis.barh(subset["Class ID"], subset["Recall (%)"])
    axis.set(title=f"{model_name}: 15 worst classes by recall", xlabel="Recall (%)", xlim=(0, 100))
WORST_CLASSES_PATH = PLOTS_DIR / "worst_classes_by_recall.png"
fig.savefig(WORST_CLASSES_PATH, dpi=180, bbox_inches="tight")
plt.show()


# Production export and Google Drive backup

Both best checkpoints are exported to ONNX and validated. YOLO26 also attempts the current Ultralytics LiteRT export, producing a `.tflite` file for future Android investigation. LiteRT export is wrapped because platform/dependency availability can vary by Colab runtime.


In [ ]:
YOLO_ONNX_PATH = Path.cwd() / "yolo26n_guardplant.onnx"
EFFICIENTNET_ONNX_PATH = Path.cwd() / "efficientnet_b0_guardplant.onnx"
YOLO_LITERT_PATH = Path.cwd() / "yolo26n_guardplant.tflite"

native_yolo_onnx = Path(
    yolo_model.export(
        format="onnx", imgsz=IMAGE_SIZE, batch=1, dynamic=False,
        simplify=False, opset=17, device="cpu"
    )
)
if native_yolo_onnx.resolve() != YOLO_ONNX_PATH.resolve():
    shutil.copy2(native_yolo_onnx, YOLO_ONNX_PATH)

efficientnet_export_model = efficientnet_model.to("cpu").eval()
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
torch.onnx.export(
    efficientnet_export_model, dummy_input, EFFICIENTNET_ONNX_PATH,
    opset_version=17, input_names=["images"], output_names=["logits"],
    dynamic_axes={"images": {0: "batch"}, "logits": {0: "batch"}},
    do_constant_folding=True, dynamo=False,
)

for model_path in (YOLO_ONNX_PATH, EFFICIENTNET_ONNX_PATH):
    graph = onnx.load(str(model_path))
    onnx.checker.check_model(graph)
    print(f"Validated ONNX: {model_path} ({file_size_mb(model_path):.2f} MB)")

if EXPORT_LITERT:
    try:
        native_litert_path = Path(
            yolo_model.export(format="litert", imgsz=IMAGE_SIZE, batch=1, device="cpu")
        )
        if native_litert_path.resolve() != YOLO_LITERT_PATH.resolve():
            shutil.copy2(native_litert_path, YOLO_LITERT_PATH)
        print(f"LiteRT/TFLite export: {YOLO_LITERT_PATH}")
    except Exception as error:
        warnings.warn(f"LiteRT export is unavailable in this runtime: {error}")


In [ ]:
required_backup_artifacts = [
    YOLO_BEST_PATH, EFFICIENTNET_BEST_PATH, CLASS_NAMES_PATH, DATASET_STATS_PATH,
    BENCHMARK_RESULTS_PATH, PER_CLASS_METRICS_PATH, TOP_CONFUSIONS_PATH,
    YOLO_ONNX_PATH, EFFICIENTNET_ONNX_PATH, TRAINING_CURVES_PATH, WORST_CLASSES_PATH,
    *CONFUSION_PLOT_PATHS.values(),
]
optional_backup_artifacts = [YOLO_HISTORY_PATH, YOLO_METADATA_PATH, EFFICIENTNET_HISTORY_PATH, YOLO_LITERT_PATH]

for artifact in required_backup_artifacts:
    print(f"Backed up: {copy_to_drive(artifact)}")
for artifact in optional_backup_artifacts:
    if Path(artifact).is_file():
        print(f"Backed up: {copy_to_drive(artifact)}")

print(f"GuardPlant artifacts are available at: {DRIVE_ARTIFACT_DIR}")
print("Notebook complete.")
